In [ ]:
# =============================================================================
# CELL 1: INSTALLATION & IMPORTS (H100 GPU OPTIMIZED)
# =============================================================================

# Install GPU-optimized packages
!pip install -q lightgbm --install-option=--gpu 2>/dev/null || pip install -q lightgbm
!pip install -q optuna scikit-learn pandas numpy matplotlib seaborn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import re
import json
import os
import gc
from datetime import datetime

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score, mean_squared_error
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ============= H100 GPU CONFIGURATION =============
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '1'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/local/cuda'
os.environ['TF_GPU_THREAD_MODE'] = 'gpu_private'
os.environ['TF_GPU_THREAD_COUNT'] = '2'

# H100-specific configuration
H100_CONFIG = {
    'lgb_batch_size': 512,       # Large batch for LightGBM
    'ann_batch_size': 1024,      # Very large batch for ANN (H100 80GB)
    'num_workers': 8,            # Parallel data loading
    'mixed_precision': True,     # Enable FP16/BF16 - H100 Tensor Cores
    'xla_compile': True,         # XLA JIT compilation
    'use_gpu_lgb': True,         # GPU histogram for LightGBM
    'prefetch_buffer': 4,        # Data prefetching
    'ann_neurons': [1024, 512, 256, 128, 64],  # Larger network for H100
}

print("="*80)
print("🏠 BANGALORE HOUSING PRICE PREDICTION PIPELINE")
print("🚀 OPTIMIZED FOR NVIDIA H100 GPU (80GB HBM3)")
print("="*80)
print(f"📅 Execution Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🎯 Target: MAPE < 20% with robust generalization")
print(f"\n⚡ H100 Optimizations:")
print(f"   • Mixed Precision: {H100_CONFIG['mixed_precision']}")
print(f"   • XLA Compilation: {H100_CONFIG['xla_compile']}")
print(f"   • ANN Batch Size: {H100_CONFIG['ann_batch_size']}")
print(f"   • Network Architecture: {H100_CONFIG['ann_neurons']}")
print("="*80)

In [ ]:
# =============================================================================
# CELL 2: MOUNT GOOGLE DRIVE & LOAD DATA
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

# ⚠️ UPDATE THIS PATH TO YOUR CSV FILE LOCATION
DATA_PATH = "/content/drive/MyDrive/YOUR_FOLDER/bangalore_housing.csv"

# Load data with robust error handling for malformed CSVs
try:
    # First attempt: standard parsing
    df_raw = pd.read_csv(DATA_PATH, low_memory=False)
except pd.errors.ParserError:
    print("⚠️ CSV has inconsistent columns, trying with error handling...")
    try:
        # Second attempt: skip bad lines
        df_raw = pd.read_csv(DATA_PATH, low_memory=False, on_bad_lines='skip')
        print("✅ Loaded with some rows skipped")
    except:
        # Third attempt: Python engine (slower but more flexible)
        df_raw = pd.read_csv(DATA_PATH, low_memory=False, engine='python', on_bad_lines='skip')
        print("✅ Loaded using Python engine")

print(f"\n📊 DATA LOADED SUCCESSFULLY!")
print(f"   Total Properties: {len(df_raw):,}")
print(f"   Total Columns: {len(df_raw.columns)}")
print(f"   Memory Usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\n📋 Column Names:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"   {i:2d}. {col}")

In [ ]:
# =============================================================================
# CELL 3: INITIAL DATA EXPLORATION - DIAGNOSTIC
# =============================================================================

print("="*80)
print("🔍 INITIAL DATA EXPLORATION - DIAGNOSTIC")
print("="*80)

print("\n📋 ALL Column Names:")
for i, col in enumerate(df_raw.columns, 1):
    sample_val = df_raw[col].dropna().iloc[0] if df_raw[col].notna().any() else "ALL NULL"
    print(f"   {i:2d}. {col:30s} -> Sample: {str(sample_val)[:60]}")

print("\n" + "="*80)
print("📊 SAMPLE ROWS (First 3):")
print("="*80)
for idx in range(min(3, len(df_raw))):
    print(f"\n--- Row {idx+1} ---")
    for col in df_raw.columns:
        val = df_raw.iloc[idx][col]
        if pd.notna(val) and str(val).strip() != '':
            print(f"   {col}: {str(val)[:80]}")

print("\n" + "="*80)
print("🔍 LOOKING FOR PRICE COLUMNS:")
print("="*80)
for col in df_raw.columns:
    sample_vals = df_raw[col].dropna().head(5).tolist()
    # Check if column might contain price
    sample_str = ' '.join([str(v) for v in sample_vals]).lower()
    if any(x in sample_str for x in ['cr', 'lakh', 'l', '₹', 'price', 'cost', 'value']):
        print(f"\n   🎯 Potential price column: '{col}'")
        print(f"      Samples: {sample_vals[:3]}")

print("\n" + "="*80)
print("🔍 LOOKING FOR SIZE/BHK COLUMNS:")
print("="*80)
for col in df_raw.columns:
    sample_vals = df_raw[col].dropna().head(5).tolist()
    sample_str = ' '.join([str(v) for v in sample_vals]).lower()
    if any(x in sample_str for x in ['bhk', 'bedroom', 'sqft', 'sq ft', 'sq.ft', 'area', 'size']):
        print(f"\n   🎯 Potential feature column: '{col}'")
        print(f"      Samples: {sample_vals[:3]}")

In [ ]:
# =============================================================================
# CELL 4: PRICE EXTRACTION & CLEANING (FIXED FOR YOUR DATA)
# =============================================================================

print("="*80)
print("💰 PRICE EXTRACTION & CLEANING")
print("="*80)

df = df_raw.copy()

def parse_price(price_str):
    """Parse price formats like '₹38.39 L', '₹1.15 Cr', '₹52.5 L'"""
    if pd.isna(price_str) or price_str == '' or 'Request' in str(price_str):
        return np.nan
    
    price_str = str(price_str).strip()
    price_str = price_str.replace('₹', '').replace(',', '').strip()
    
    try:
        # Handle Crores -> convert to Lakhs
        if 'Cr' in price_str or 'cr' in price_str:
            value = float(re.findall(r'[\d.]+', price_str)[0])
            return value * 100  # 1 Cr = 100 Lakhs
        # Handle Lakhs
        elif 'L' in price_str or 'l' in price_str:
            value = float(re.findall(r'[\d.]+', price_str)[0])
            return value
        else:
            # Try to parse as raw number
            value = float(re.findall(r'[\d.]+', price_str)[0])
            if value > 10000:  # Probably in raw rupees
                return value / 100000  # Convert to Lakhs
            return value
    except:
        return np.nan

# Use 'price_value' column (the clean one!)
print(f"\n🔍 Using price column: 'price_value'")
print(f"   Sample values: {df['price_value'].dropna().head(5).tolist()}")

df['price_lakhs'] = df['price_value'].apply(parse_price)

# Show parsing results
print(f"\n📊 Price Parsing Results:")
for idx in range(min(5, len(df))):
    orig = df['price_value'].iloc[idx]
    parsed = df['price_lakhs'].iloc[idx]
    print(f"   '{orig}' -> ₹{parsed:.2f} L" if pd.notna(parsed) else f"   '{orig}' -> NaN")

valid_prices = df['price_lakhs'].dropna()
print(f"\n📊 Price Statistics (in Lakhs):")
print(f"   Valid prices: {len(valid_prices):,} ({len(valid_prices)/len(df)*100:.1f}%)")
print(f"   Min: ₹{valid_prices.min():.2f} L | Max: ₹{valid_prices.max():.2f} L")
print(f"   Mean: ₹{valid_prices.mean():.2f} L | Median: ₹{valid_prices.median():.2f} L")

In [ ]:
# =============================================================================
# CELL 5: FEATURE EXTRACTION (FIXED FOR YOUR DATA)
# =============================================================================

print("="*80)
print("🏗️ FEATURE EXTRACTION")
print("="*80)

# Your data already has clean columns! Let's use them directly.

# 1. BHK - already numeric in 'bhk' column
print("\n📊 BHK Column:")
print(f"   Sample values: {df['bhk'].dropna().head(10).tolist()}")
df['bhk'] = pd.to_numeric(df['bhk'], errors='coerce')
print(f"   Valid BHK: {df['bhk'].notna().sum()} ({df['bhk'].notna().sum()/len(df)*100:.1f}%)")
print(f"   Distribution: {df['bhk'].value_counts().sort_index().to_dict()}")

# 2. Size - already numeric in 'sqft' column
print("\n📊 SQFT Column:")
print(f"   Sample values: {df['sqft'].dropna().head(10).tolist()}")
df['size_sqft'] = pd.to_numeric(df['sqft'], errors='coerce')
print(f"   Valid sqft: {df['size_sqft'].notna().sum()} ({df['size_sqft'].notna().sum()/len(df)*100:.1f}%)")
print(f"   Range: {df['size_sqft'].min():.0f} - {df['size_sqft'].max():.0f} sq.ft")

# 3. Image Count - already numeric
print("\n📊 Image Count Column:")
df['image_count'] = pd.to_numeric(df['image_count'], errors='coerce')
print(f"   Valid: {df['image_count'].notna().sum()} ({df['image_count'].notna().sum()/len(df)*100:.1f}%)")

# 4. Location - extract from page_title or title
print("\n📊 Location Extraction:")
def extract_location(row):
    # Try page_title first: "New Projects in Bagalur, Hosur | ..."
    if pd.notna(row.get('page_title')):
        text = str(row['page_title'])
        # Pattern: "... in LOCATION | ..." or "... in LOCATION, CITY | ..."
        match = re.search(r'in\s+([^|]+)', text)
        if match:
            loc = match.group(1).strip()
            # Take first part before comma if it's a location
            parts = loc.split(',')
            if len(parts) >= 1:
                return parts[0].strip()[:40]
    
    # Fallback to title
    if pd.notna(row.get('title')):
        text = str(row['title'])
        match = re.search(r'in\s+([^,|]+)', text)
        if match:
            return match.group(1).strip()[:40]
    
    return 'Unknown'

df['location'] = df.apply(extract_location, axis=1)
loc_counts = df['location'].value_counts()
print(f"   Unique locations: {len(loc_counts)}")
print(f"   Top 5: {loc_counts.head(5).to_dict()}")

# 5. Bathrooms (if available)
if 'ctx_bathrooms' in df.columns:
    # Try to extract number from ctx_bathrooms text
    def extract_bathrooms(text):
        if pd.isna(text):
            return np.nan
        match = re.search(r'(\d+)\s*(?:bath|bathroom)', str(text).lower())
        if match:
            return int(match.group(1))
        return np.nan
    df['bathrooms'] = df['ctx_bathrooms'].apply(extract_bathrooms)
    print(f"\n📊 Bathrooms: {df['bathrooms'].notna().sum()} valid values")

print("\n" + "="*80)
print("📋 FEATURE SUMMARY:")
print("="*80)
print(f"   ✓ bhk: {df['bhk'].notna().sum()} valid")
print(f"   ✓ size_sqft: {df['size_sqft'].notna().sum()} valid")
print(f"   ✓ image_count: {df['image_count'].notna().sum()} valid")
print(f"   ✓ location: {(df['location'] != 'Unknown').sum()} valid")
print(f"   ✓ price_lakhs: {df['price_lakhs'].notna().sum()} valid")
print("="*80)

In [ ]:
# =============================================================================
# CELL 6: EXPLORATORY DATA ANALYSIS
# =============================================================================

print("="*80)
print("📊 EXPLORATORY DATA ANALYSIS")
print("="*80)

df_valid = df[df['price_lakhs'].notna()].copy()
print(f"\n📈 Working with {len(df_valid):,} properties with valid prices")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Price Distribution
ax1 = axes[0, 0]
ax1.hist(df_valid['price_lakhs'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(df_valid['price_lakhs'].median(), color='red', linestyle='--', label=f'Median: ₹{df_valid["price_lakhs"].median():.1f}L')
ax1.set_xlabel('Price (Lakhs)')
ax1.set_ylabel('Frequency')
ax1.set_title('Price Distribution', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log Price
ax2 = axes[0, 1]
df_valid['log_price'] = np.log1p(df_valid['price_lakhs'])
ax2.hist(df_valid['log_price'], bins=50, color='coral', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Log(Price + 1)')
ax2.set_title('Log Price Distribution', fontweight='bold')
ax2.grid(True, alpha=0.3)

# BHK Distribution
ax3 = axes[0, 2]
if 'bhk' in df_valid.columns and df_valid['bhk'].notna().any():
    bhk_counts = df_valid['bhk'].value_counts().sort_index()
    ax3.bar(bhk_counts.index.astype(str), bhk_counts.values, color='teal', alpha=0.7)
ax3.set_xlabel('BHK')
ax3.set_title('BHK Distribution', fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Price vs Size
ax4 = axes[1, 0]
if 'size_sqft' in df_valid.columns and df_valid['size_sqft'].notna().any():
    valid_size = df_valid[df_valid['size_sqft'].notna() & (df_valid['size_sqft'] > 0)]
    ax4.scatter(valid_size['size_sqft'], valid_size['price_lakhs'], alpha=0.5, s=20)
ax4.set_xlabel('Size (sq.ft)')
ax4.set_ylabel('Price (Lakhs)')
ax4.set_title('Price vs Size', fontweight='bold')
ax4.grid(True, alpha=0.3)

# Price by BHK
ax5 = axes[1, 1]
if 'bhk' in df_valid.columns and df_valid['bhk'].notna().any():
    bhk_valid = df_valid[df_valid['bhk'].notna() & (df_valid['bhk'] <= 6)]
    bhk_valid.boxplot(column='price_lakhs', by='bhk', ax=ax5)
ax5.set_xlabel('BHK')
ax5.set_ylabel('Price (Lakhs)')
ax5.set_title('Price by BHK', fontweight='bold')
plt.suptitle('')

# Top Locations
ax6 = axes[1, 2]
if 'location' in df_valid.columns:
    top_locs = df_valid.groupby('location')['price_lakhs'].agg(['mean', 'count']).reset_index()
    top_locs = top_locs[top_locs['count'] >= 5].nlargest(10, 'mean')
    ax6.barh(range(len(top_locs)), top_locs['mean'], color='darkgreen', alpha=0.7)
    ax6.set_yticks(range(len(top_locs)))
    ax6.set_yticklabels(top_locs['location'].str[:20], fontsize=9)
    ax6.invert_yaxis()
ax6.set_xlabel('Avg Price (Lakhs)')
ax6.set_title('Top 10 Locations', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 7: DATA CLEANING & OUTLIER REMOVAL
# =============================================================================

print("="*80)
print("🧹 DATA CLEANING & OUTLIER REMOVAL")
print("="*80)

df_clean = df_valid.copy()
print(f"\n📊 Starting with {len(df_clean):,} properties")

# Price bounds (relaxed for small datasets)
min_price, max_price = 5, 5000
df_clean = df_clean[(df_clean['price_lakhs'] >= min_price) & (df_clean['price_lakhs'] <= max_price)]
print(f"   After price bounds ({min_price}L - {max_price}L): {len(df_clean):,}")

# BHK bounds
if 'bhk' in df_clean.columns:
    df_clean = df_clean[(df_clean['bhk'].isna()) | ((df_clean['bhk'] >= 1) & (df_clean['bhk'] <= 10))]
    print(f"   After BHK bounds: {len(df_clean):,}")

# Size bounds
if 'size_sqft' in df_clean.columns:
    df_clean = df_clean[(df_clean['size_sqft'].isna()) | ((df_clean['size_sqft'] >= 200) & (df_clean['size_sqft'] <= 50000))]
    print(f"   After size bounds: {len(df_clean):,}")

# IQR outlier removal - ONLY if we have enough data
if len(df_clean) > 50:
    Q1 = df_clean['price_lakhs'].quantile(0.05)
    Q3 = df_clean['price_lakhs'].quantile(0.95)
    IQR = Q3 - Q1
    lower_bound = max(Q1 - 1.5 * IQR, min_price)
    upper_bound = min(Q3 + 1.5 * IQR, max_price)
    df_clean = df_clean[(df_clean['price_lakhs'] >= lower_bound) & (df_clean['price_lakhs'] <= upper_bound)]
    print(f"   After IQR outlier removal: {len(df_clean):,}")
else:
    print(f"   Skipping IQR removal (small dataset: {len(df_clean)} samples)")

# Location filtering - ADAPTIVE threshold based on dataset size
if 'location' in df_clean.columns:
    loc_counts = df_clean['location'].value_counts()
    
    # Adaptive threshold: for small datasets, use 1 (no filtering)
    if len(df_clean) < 50:
        min_loc_count = 1  # No filtering for very small datasets
    elif len(df_clean) < 200:
        min_loc_count = 2
    else:
        min_loc_count = 3
    
    valid_locs = loc_counts[loc_counts >= min_loc_count].index
    df_before = len(df_clean)
    df_clean = df_clean[df_clean['location'].isin(valid_locs)]
    print(f"   After location filtering (min {min_loc_count}): {len(df_clean):,}")
    
    if len(df_clean) == 0:
        # Fallback: keep all if filtering removes everything
        print(f"   ⚠️ Location filter removed all data! Reverting...")
        df_clean = df_valid[(df_valid['price_lakhs'] >= min_price) & 
                           (df_valid['price_lakhs'] <= max_price)].copy()
        print(f"   Restored: {len(df_clean):,}")

print(f"\n✅ Final cleaned dataset: {len(df_clean):,} properties")

# Warning if dataset is very small
if len(df_clean) < 30:
    print(f"\n⚠️ WARNING: Only {len(df_clean)} samples - model may have limited accuracy")
    print("   Consider scraping more data for better predictions")

In [ ]:
# =============================================================================
# CELL 8: FEATURE ENGINEERING
# =============================================================================

print("="*80)
print("🔧 FEATURE ENGINEERING")
print("="*80)

df_feat = df_clean.copy()

# Size per BHK
if 'size_sqft' in df_feat.columns and 'bhk' in df_feat.columns:
    valid_both = df_feat['size_sqft'].notna() & df_feat['bhk'].notna()
    df_feat.loc[valid_both, 'size_per_bhk'] = df_feat.loc[valid_both, 'size_sqft'] / (df_feat.loc[valid_both, 'bhk'] + 0.1)

# Location encoding prep
if 'location' in df_feat.columns:
    df_feat['location_clean'] = df_feat['location'].str.lower().str.strip()
    df_feat['location_clean'] = df_feat['location_clean'].str.replace(r'[^a-z0-9\s]', '', regex=True)

# Log transform
if 'size_sqft' in df_feat.columns:
    df_feat['log_size'] = np.log1p(df_feat['size_sqft'])

# BHK category
if 'bhk' in df_feat.columns:
    df_feat['bhk_category'] = pd.cut(
        df_feat['bhk'], 
        bins=[0, 1, 2, 3, 4, float('inf')],
        labels=['1BHK', '2BHK', '3BHK', '4BHK', '5+BHK']
    )

# Image quality proxy
if 'image_count' in df_feat.columns:
    df_feat['has_many_images'] = (df_feat['image_count'] >= 10).astype(int)

print(f"\n📊 Dataset shape: {df_feat.shape}")
print("✅ Feature engineering complete!")

In [ ]:
# =============================================================================
# CELL 9: TRAIN-TEST SPLIT (LEAKAGE-FREE)
# =============================================================================

print("="*80)
print("🎯 TRAIN-TEST SPLIT (LEAKAGE-FREE)")
print("="*80)

TARGET = 'price_lakhs'

# Debug: Show available columns
print(f"\n🔍 Available columns in df_feat:")
print(f"   {list(df_feat.columns)}")

# Select features - more flexible approach
numeric_features = ['bhk', 'size_sqft', 'bathrooms', 'image_count', 'log_size', 'size_per_bhk']
categorical_features = ['location_clean', 'bhk_category']

# Check which features actually exist and have data
print(f"\n📊 Feature availability check:")
available_numeric = []
for f in numeric_features:
    if f in df_feat.columns:
        count = df_feat[f].notna().sum()
        print(f"   ✓ {f}: {count} non-null values")
        if count > 10:  # Lower threshold
            available_numeric.append(f)
    else:
        print(f"   ✗ {f}: NOT FOUND")

available_categorical = []
for f in categorical_features:
    if f in df_feat.columns:
        count = df_feat[f].notna().sum()
        print(f"   ✓ {f}: {count} non-null values")
        if count > 10:  # Lower threshold
            available_categorical.append(f)
    else:
        print(f"   ✗ {f}: NOT FOUND")

numeric_features = available_numeric
categorical_features = available_categorical

# If no features found, try to use ANY numeric columns
if len(numeric_features) == 0:
    print("\n⚠️ No predefined numeric features found! Searching for alternatives...")
    for col in df_feat.columns:
        if col != TARGET and df_feat[col].dtype in ['int64', 'float64']:
            if df_feat[col].notna().sum() > 10:
                numeric_features.append(col)
                print(f"   + Added: {col}")

all_features = numeric_features + categorical_features
print(f"\n📋 FINAL Selected Features: {all_features}")

if len(all_features) == 0:
    raise ValueError("❌ No features available! Check data cleaning steps.")

df_model = df_feat[all_features + [TARGET]].copy()

for col in numeric_features:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

df_model = df_model[df_model[TARGET].notna()]

print(f"\n📊 df_model shape: {df_model.shape}")

X = df_model.drop(TARGET, axis=1)
y = df_model[TARGET]

print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")

if X.shape[1] == 0:
    raise ValueError("❌ X has no features! Check preprocessing.")

# Split FIRST (before encoding)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, shuffle=True
)

print(f"\n✅ Train-Test Split Complete:")
print(f"   Training: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"   Test: {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")
print(f"   Features: {list(X_train.columns)}")

In [ ]:
# =============================================================================
# CELL 10: ENCODING & IMPUTATION (LEAKAGE-FREE)
# =============================================================================

print("="*80)
print("🔄 ENCODING & IMPUTATION")
print("="*80)

encoders = {}
imputers = {}

# Make copies to avoid SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

print(f"\n📊 Starting with {X_train.shape[1]} features: {list(X_train.columns)}")

# Impute numeric features
print("\n📊 Imputing numeric features...")
for col in numeric_features:
    if col in X_train.columns:
        median_val = X_train[col].median()
        if pd.isna(median_val):
            median_val = 0  # Fallback
        imputers[col] = median_val
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)
        print(f"   ✓ {col}: median = {median_val:.2f}")

# Target encoding for categorical (only if we have categorical features)
if len(categorical_features) > 0:
    print("\n📊 Target encoding categorical features...")
    for col in categorical_features:
        if col in X_train.columns:
            target_means = y_train.groupby(X_train[col]).mean()
            global_mean = y_train.mean()
            encoders[col] = {'target_means': target_means, 'global_mean': global_mean}
            
            X_train[f'{col}_encoded'] = X_train[col].map(target_means).fillna(global_mean)
            X_test[f'{col}_encoded'] = X_test[col].map(target_means).fillna(global_mean)
            
            X_train = X_train.drop(col, axis=1)
            X_test = X_test.drop(col, axis=1)
            print(f"   ✓ {col}: {len(target_means)} unique values -> {col}_encoded")
else:
    print("\n📊 No categorical features to encode")

print(f"\n✅ Final features ({X_train.shape[1]}): {list(X_train.columns)}")

# Final validation
if X_train.shape[1] == 0:
    raise ValueError("❌ No features remaining after encoding!")

print(f"\n📊 Final shapes:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")

In [ ]:
# =============================================================================
# CELL 11: BASELINE LIGHTGBM (H100 GPU ACCELERATED)
# =============================================================================

print("="*80)
print("🚀 BASELINE LIGHTGBM (H100 GPU ACCELERATED)")
print("="*80)

# H100 GPU-optimized parameters
baseline_params = {
    'objective': 'regression',
    'metric': 'mape',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'max_depth': 6,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}

# Enable GPU if available
if H100_CONFIG['use_gpu_lgb']:
    try:
        baseline_params.update({
            'device': 'gpu',
            'gpu_platform_id': 0,
            'gpu_device_id': 0,
            'gpu_use_dp': False,
        })
        print("\n⚡ GPU acceleration ENABLED for LightGBM")
    except:
        print("\n⚠️ GPU not available, using CPU")

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

print("\n🏋️ Training baseline model...")
baseline_model = lgb.train(
    baseline_params,
    train_data,
    num_boost_round=2000,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
)

# Evaluate
y_train_pred_baseline = baseline_model.predict(X_train, num_iteration=baseline_model.best_iteration)
y_test_pred_baseline = baseline_model.predict(X_test, num_iteration=baseline_model.best_iteration)

train_mape_baseline = mean_absolute_percentage_error(y_train, y_train_pred_baseline) * 100
test_mape_baseline = mean_absolute_percentage_error(y_test, y_test_pred_baseline) * 100
train_r2_baseline = r2_score(y_train, y_train_pred_baseline)
test_r2_baseline = r2_score(y_test, y_test_pred_baseline)

print(f"\n📊 BASELINE RESULTS:")
print(f"   Train MAPE: {train_mape_baseline:.2f}% | R²: {train_r2_baseline:.4f}")
print(f"   Test MAPE: {test_mape_baseline:.2f}% | R²: {test_r2_baseline:.4f}")
print(f"   Best Iteration: {baseline_model.best_iteration}")

In [ ]:
# =============================================================================
# CELL 12: OPTUNA HYPERPARAMETER TUNING (H100 ACCELERATED)
# =============================================================================

print("="*80)
print("🔬 OPTUNA HYPERPARAMETER TUNING (H100 ACCELERATED)")
print("="*80)

# Adjust trials based on dataset size
n_samples = len(X_train)
if n_samples < 50:
    N_TRIALS = 20  # Small dataset - fewer trials
    N_FOLDS = 3    # Fewer folds for small data
    print(f"\n⚠️ Small dataset ({n_samples} samples) - using {N_TRIALS} trials, {N_FOLDS}-fold CV")
elif n_samples < 200:
    N_TRIALS = 50
    N_FOLDS = 4
else:
    N_TRIALS = 100
    N_FOLDS = 5

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mape',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'feature_pre_filter': False,  # CRITICAL: Allow dynamic min_child_samples
        'num_leaves': trial.suggest_int('num_leaves', 10, min(100, n_samples//2)),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.95),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.95),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, max(5, n_samples//10)),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    
    if H100_CONFIG['use_gpu_lgb']:
        params.update({'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0})
    
    try:
        cv_results = lgb.cv(
            params, train_data,
            num_boost_round=500,  # Reduced for small datasets
            nfold=N_FOLDS,
            stratified=False,
            shuffle=True,
            callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)],
            seed=RANDOM_STATE
        )
        return min(cv_results['valid mape-mean'])
    except Exception as e:
        # Return a high value if CV fails (e.g., not enough samples per fold)
        print(f"   Trial failed: {str(e)[:50]}")
        return 1.0  # 100% MAPE as penalty

print(f"\n🔍 Running {N_TRIALS} trials...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True, gc_after_trial=True)

best_params = study.best_params
print(f"\n🏆 BEST PARAMETERS:")
for k, v in best_params.items():
    print(f"   {k}: {v:.6f}" if isinstance(v, float) else f"   {k}: {v}")
print(f"\n   Best CV MAPE: {study.best_value*100:.2f}%")

In [ ]:
# =============================================================================
# CELL 13: TRAIN OPTIMIZED LIGHTGBM
# =============================================================================

print("="*80)
print("🚀 TRAINING OPTIMIZED LIGHTGBM")
print("="*80)

optimized_params = {
    'objective': 'regression',
    'metric': 'mape',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'feature_pre_filter': False,  # CRITICAL: Allow dynamic min_child_samples
    **best_params
}

if H100_CONFIG['use_gpu_lgb']:
    optimized_params.update({'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0})

# Recreate datasets to avoid state issues
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

print("\n🏋️ Training optimized model...")
optimized_model = lgb.train(
    optimized_params,
    train_data,
    num_boost_round=1000,  # Reduced for small datasets
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

y_train_pred_opt = optimized_model.predict(X_train, num_iteration=optimized_model.best_iteration)
y_test_pred_opt = optimized_model.predict(X_test, num_iteration=optimized_model.best_iteration)

train_mape_opt = mean_absolute_percentage_error(y_train, y_train_pred_opt) * 100
test_mape_opt = mean_absolute_percentage_error(y_test, y_test_pred_opt) * 100
train_r2_opt = r2_score(y_train, y_train_pred_opt)
test_r2_opt = r2_score(y_test, y_test_pred_opt)
train_mae_opt = mean_absolute_error(y_train, y_train_pred_opt)
test_mae_opt = mean_absolute_error(y_test, y_test_pred_opt)

print(f"\n📊 OPTIMIZED MODEL RESULTS:")
print(f"   Train MAPE: {train_mape_opt:.2f}% | R²: {train_r2_opt:.4f} | MAE: ₹{train_mae_opt:.2f}L")
print(f"   Test MAPE: {test_mape_opt:.2f}% | R²: {test_r2_opt:.4f} | MAE: ₹{test_mae_opt:.2f}L")
print(f"\n📈 Improvement: {test_mape_baseline - test_mape_opt:+.2f}% MAPE")

In [ ]:
# =============================================================================
# CELL 14: FEATURE IMPORTANCE
# =============================================================================

print("="*80)
print("📊 FEATURE IMPORTANCE")
print("="*80)

importance_df = pd.DataFrame({
    'Feature': optimized_model.feature_name(),
    'Importance': optimized_model.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=False)

print("\n🏆 Feature Importance:")
for _, row in importance_df.iterrows():
    pct = row['Importance'] / importance_df['Importance'].sum() * 100
    print(f"   {row['Feature']:25s}: {row['Importance']:>10.2f} ({pct:>5.1f}%)")

plt.figure(figsize=(10, 6))
plt.barh(range(len(importance_df)), importance_df['Importance'], color='steelblue', alpha=0.8)
plt.yticks(range(len(importance_df)), importance_df['Feature'])
plt.xlabel('Importance (Gain)')
plt.title('Feature Importance (LightGBM)', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 15: LIGHTGBM VISUALIZATION
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Train: Actual vs Predicted
axes[0, 0].scatter(y_train, y_train_pred_opt, alpha=0.5, s=20, color='steelblue')
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual (₹ Lakhs)')
axes[0, 0].set_ylabel('Predicted (₹ Lakhs)')
axes[0, 0].set_title(f'Train: Actual vs Predicted\nMAPE: {train_mape_opt:.2f}%', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Test: Actual vs Predicted
axes[0, 1].scatter(y_test, y_test_pred_opt, alpha=0.5, s=20, color='green')
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual (₹ Lakhs)')
axes[0, 1].set_ylabel('Predicted (₹ Lakhs)')
axes[0, 1].set_title(f'Test: Actual vs Predicted\nMAPE: {test_mape_opt:.2f}%', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Residuals
axes[1, 0].scatter(y_train_pred_opt, y_train - y_train_pred_opt, alpha=0.5, s=20, color='steelblue')
axes[1, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].set_title('Train: Residuals', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].scatter(y_test_pred_opt, y_test - y_test_pred_opt, alpha=0.5, s=20, color='green')
axes[1, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Test: Residuals', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 16: ANN MODEL (SIMPLIFIED FOR SMALL DATASETS)
# =============================================================================

!pip install -q tensorflow

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print("="*80)
print("🧠 ANN MODEL (OPTIMIZED)")
print("="*80)

# Clear any existing sessions to avoid CUDA context issues
keras.backend.clear_session()
gc.collect()

# ============= SIMPLIFIED GPU SETUP (avoids CUDA context errors) =============
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU Available: {len(gpus)} GPU(s)")
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass  # Already set

print(f"\nTensorFlow: {tf.__version__}")
print(f"CUDA: {tf.test.is_built_with_cuda()}")

# Prepare data
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train).astype('float32')
X_test_scaled = scaler.transform(X_test).astype('float32')
y_train_ann = y_train.values.astype('float32')
y_test_ann = y_test.values.astype('float32')

# Adaptive architecture for small datasets
input_dim = X_train_scaled.shape[1]
n_samples = len(X_train_scaled)

# Scale network size based on dataset
if n_samples < 50:
    neurons = [64, 32, 16]
    batch_size = min(16, n_samples)
    dropout_rate = 0.3
    print(f"\n⚠️ Small dataset ({n_samples} samples) - using smaller network")
elif n_samples < 200:
    neurons = [128, 64, 32]
    batch_size = min(32, n_samples)
    dropout_rate = 0.3
else:
    neurons = [256, 128, 64, 32]
    batch_size = min(128, n_samples)
    dropout_rate = 0.4

print(f"   Network: {neurons}")
print(f"   Batch size: {batch_size}")

# Build model - simpler architecture
ann_model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(neurons[0], activation='relu', kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.BatchNormalization(),
    layers.Dropout(dropout_rate),
])

for n in neurons[1:]:
    ann_model.add(layers.Dense(n, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-3)))
    ann_model.add(layers.BatchNormalization())
    ann_model.add(layers.Dropout(dropout_rate * 0.7))

ann_model.add(layers.Dense(1, activation='linear'))

# Compile with standard settings
ann_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',  # MSE is more stable than MAPE for small datasets
    metrics=['mae']
)

print(f"\n📋 Model Summary:")
ann_model.summary()

# Callbacks
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)

# Train
print(f"\n🏋️ Training with batch_size={batch_size}...")
history = ann_model.fit(
    X_train_scaled, y_train_ann,
    validation_data=(X_test_scaled, y_test_ann),
    epochs=300,
    batch_size=batch_size,
    callbacks=[early_stop, reduce_lr],
    verbose=2
)

print(f"\n✅ Training complete! Epochs: {len(history.history['loss'])}")

In [ ]:
# =============================================================================
# CELL 17: ANN EVALUATION
# =============================================================================

print("="*80)
print("📊 ANN EVALUATION")
print("="*80)

y_train_pred_ann = ann_model.predict(X_train_scaled, verbose=0, batch_size=H100_CONFIG['ann_batch_size']).flatten()
y_test_pred_ann = ann_model.predict(X_test_scaled, verbose=0, batch_size=H100_CONFIG['ann_batch_size']).flatten()

train_mape_ann = mean_absolute_percentage_error(y_train_ann, y_train_pred_ann) * 100
test_mape_ann = mean_absolute_percentage_error(y_test_ann, y_test_pred_ann) * 100
train_r2_ann = r2_score(y_train_ann, y_train_pred_ann)
test_r2_ann = r2_score(y_test_ann, y_test_pred_ann)
train_mae_ann = mean_absolute_error(y_train_ann, y_train_pred_ann)
test_mae_ann = mean_absolute_error(y_test_ann, y_test_pred_ann)

print(f"\n🎯 ANN Results:")
print(f"   Train MAPE: {train_mape_ann:.2f}% | R²: {train_r2_ann:.4f} | MAE: ₹{train_mae_ann:.2f}L")
print(f"   Test MAPE: {test_mape_ann:.2f}% | R²: {test_r2_ann:.4f} | MAE: ₹{test_mae_ann:.2f}L")

# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training History: Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training History: MAE', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 18: MODEL COMPARISON
# =============================================================================

print("="*80)
print("🏆 MODEL COMPARISON: LightGBM vs ANN")
print("="*80)

comparison = pd.DataFrame({
    'Model': ['LightGBM', 'ANN'],
    'Train MAPE': [f'{train_mape_opt:.2f}%', f'{train_mape_ann:.2f}%'],
    'Test MAPE': [f'{test_mape_opt:.2f}%', f'{test_mape_ann:.2f}%'],
    'Train R²': [f'{train_r2_opt:.4f}', f'{train_r2_ann:.4f}'],
    'Test R²': [f'{test_r2_opt:.4f}', f'{test_r2_ann:.4f}'],
})
display(comparison)

winner = 'LightGBM' if test_mape_opt < test_mape_ann else 'ANN'
winner_mape = min(test_mape_opt, test_mape_ann)
winner_r2 = test_r2_opt if test_mape_opt < test_mape_ann else test_r2_ann

print(f"\n🏆 WINNER: {winner}")
print(f"   Test MAPE: {winner_mape:.2f}%")
print(f"   Test R²: {winner_r2:.4f}")
print(f"   Target MAPE < 20%: {'✅ ACHIEVED' if winner_mape < 20 else '❌ NOT ACHIEVED'}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(2)
width = 0.35

axes[0].bar(x - width/2, [train_mape_opt, train_mape_ann], width, label='Train', color='steelblue')
axes[0].bar(x + width/2, [test_mape_opt, test_mape_ann], width, label='Test', color='coral')
axes[0].axhline(y=20, color='red', linestyle='--', alpha=0.5, label='Target: 20%')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['LightGBM', 'ANN'])
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('MAPE Comparison', fontweight='bold')
axes[0].legend()

axes[1].bar(x - width/2, [train_r2_opt, train_r2_ann], width, label='Train', color='steelblue')
axes[1].bar(x + width/2, [test_r2_opt, test_r2_ann], width, label='Test', color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['LightGBM', 'ANN'])
axes[1].set_ylabel('R² Score')
axes[1].set_title('R² Comparison', fontweight='bold')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 19: ERROR ANALYSIS & BIAS CORRECTION
# =============================================================================

print("="*80)
print("🔍 ERROR ANALYSIS & BIAS CORRECTION")
print("="*80)

y_pred_final = y_test_pred_opt if winner == 'LightGBM' else y_test_pred_ann
y_actual = y_test.values if winner == 'LightGBM' else y_test_ann

pct_errors = ((y_pred_final - y_actual) / y_actual) * 100

print(f"\n📊 Error Distribution ({winner}):")
print(f"   Mean Error: {pct_errors.mean():.2f}%")
print(f"   Median Error: {np.median(pct_errors):.2f}%")
print(f"   Std Dev: {pct_errors.std():.2f}%")

# Bias correction
mean_bias = pct_errors.mean()
if abs(mean_bias) > 2:
    correction_factor = 1 - (mean_bias / 100)
    print(f"\n⚠️ Bias Detected: {mean_bias:.2f}%")
    print(f"   Correction Factor: {correction_factor:.4f}")
    
    y_pred_corrected = y_pred_final * correction_factor
    corrected_mape = mean_absolute_percentage_error(y_actual, y_pred_corrected) * 100
    print(f"\n   Corrected MAPE: {corrected_mape:.2f}%")
else:
    correction_factor = 1.0
    corrected_mape = winner_mape
    print(f"\n✅ No significant bias (mean: {mean_bias:.2f}%)")

# Error distribution plot
plt.figure(figsize=(10, 5))
plt.hist(pct_errors, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--', lw=2, label='Zero Error')
plt.axvline(x=mean_bias, color='green', linestyle='-', lw=2, label=f'Mean: {mean_bias:.2f}%')
plt.xlabel('Percentage Error (%)')
plt.ylabel('Frequency')
plt.title('Error Distribution', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 20: FINAL SUMMARY & SAVE MODELS
# =============================================================================

print("="*80)
print("📋 FINAL SUMMARY")
print("="*80)

print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║           BANGALORE HOUSING PRICE PREDICTION - H100 OPTIMIZED               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  📊 Dataset: {len(df_feat):,} properties | {len(X_train.columns)} features                            ║
║  🎯 Winner: {winner:10s} | Test MAPE: {corrected_mape:.2f}%                            ║
║  ✅ Target < 20%: {'ACHIEVED' if corrected_mape < 20 else 'NOT ACHIEVED'}                                              ║
║                                                                              ║
║  ⚡ H100 Optimizations Used:                                                 ║
║     • Mixed Precision (FP16): {H100_CONFIG['mixed_precision']}                                   ║
║     • XLA Compilation: {H100_CONFIG['xla_compile']}                                       ║
║     • ANN Batch Size: {H100_CONFIG['ann_batch_size']}                                           ║
║     • Network: {H100_CONFIG['ann_neurons']}                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

# Save models
print("\n💾 SAVING MODELS...")
optimized_model.save_model('/content/drive/MyDrive/bangalore_lgbm_h100.txt')
print("   ✓ LightGBM saved")

ann_model.save('/content/drive/MyDrive/bangalore_ann_h100.h5')
print("   ✓ ANN saved")

import pickle
artifacts = {
    'encoders': encoders,
    'imputers': imputers,
    'scaler': scaler,
    'correction_factor': correction_factor,
    'feature_columns': list(X_train.columns),
    'H100_CONFIG': H100_CONFIG
}
with open('/content/drive/MyDrive/bangalore_artifacts_h100.pkl', 'wb') as f:
    pickle.dump(artifacts, f)
print("   ✓ Artifacts saved")

print("\n" + "="*80)
print("✅ PIPELINE COMPLETE!")
print("="*80)